# Faz 3.1 — Kural Tabanlı (Motorlu) İhlal Ekleme

**LLM yok.** Baseline'a deterministik olarak ihlal ekliyoruz ve viewer'da görüyoruz.

| | Anlam | IFC |
|--|--|--|
| 🔴 `violation` | Kuralı bozan gerçek değişim | değişir |
| 🟡 `decoy` | 'İhlal' denmiş ama uygun | değişmez |
| 🟢 `compliant` | Kural bozmayan yeni ekleme | eklenir |

Çıktı: `data/violated_ifc/<ad>.ifc` + `<ad>.meta.json` (ground-truth etiketler).
Kural seti: [`docs/kural_seti.md`](../docs/kural_seti.md) — R1 kapı genişliği, R2 pencere/taban oranı, R3 kat yüksekliği.

## 0) Kurulum

In [ ]:
import sys, glob, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = REPO / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
print('src:', SRC)

## 1) Baseline kurallara uygun mu?
Kural tabanlı baseline **tüm kuralları sağlamalı** (ihlal sayısı 0). Önce bunu doğruluyoruz.

In [ ]:
from ifc_gen.baseline.rule_based import build_baseline
from ifc_gen.inject import rules
from viewer.model import load_viewer_model

_, baseline_path, _ = build_baseline()
vmb = load_viewer_model(str(baseline_path))
ihlaller = rules.violations_only(vmb)
print('Baseline:', baseline_path.name)
print('Baseline ihlal sayısı (0 olmalı):', len(ihlaller))
for f in rules.detect(vmb):
    if f.rule == 'R2_window_floor_ratio':
        print(f'  {vmb.elements[f.ekey].name}: {f.detail}')

## 2) Manuel ihlal ekleme — adım adım
`ViolationInjector` ile her ihlali tek tek uyguluyoruz:

In [ ]:
from ifc_gen.inject.rule_based import ViolationInjector

inj = ViolationInjector()
inj.narrow_door('Kapi-IC1', new_width=0.70)   # R1 🔴 iç kapı 0.70 < 0.90 m
inj.remove_window('Pencere-R2')               # R2 🔴 orta odanın penceresi gider
inj.mark_decoy('Kapi-Giris')                  # 🟡 uygun ama 'ihlal' etiketli
inj.add_compliant_column((6.0, 4.0))          # 🟢 kural bozmayan kolon
print('Uygulanan mutasyonlar:')
for m in inj._mutations: print('  -', m)

## 3) Üret → violated IFC + meta.json

In [ ]:
model, violated_path, meta = inj.build()
print('Violated IFC:', violated_path.name)
print('Etiket sayıları:', meta['counts'])
print()
for a in meta['annotations']:
    print(f"  {a['status']:10s} {a['rule']:24s} {a['name']:12s} | {a['detail']}")

## 4) Doğrulama — enjekte edilen ihlaller gerçekten kuralı bozuyor mu?
Tespit (`rules.detect`) **tam olarak gerçek ihlalleri** bulmalı; decoy 🟡 ve compliant 🟢 ihlal olarak görünmemeli. Ground-truth ↔ tespit tutarlılığı.

In [ ]:
vm = load_viewer_model(str(violated_path))
tespit = rules.violations_only(vm)
print('Tespit edilen ihlaller:')
for f in tespit:
    print(f'  {f.rule:24s} {vm.elements[f.ekey].name:10s} {f.detail}')

gt = {a['ekey'] for a in meta['annotations'] if a['status']=='violation'}
det = {f.ekey for f in tespit}
print()
print('Ground-truth ihlal ekey =', gt)
print('Tespit edilen ekey      =', det)
print('EŞLEŞME:', 'EVET ✅' if gt == det else 'HAYIR ⚠️')

## 5) Görselleştir — 🔴/🟡/🟢
Statik önizleme (her ortam) + tek satır interaktif `view(...)`.

In [ ]:
import matplotlib.pyplot as plt
from viewer import static, load_labels
labels = load_labels(str(violated_path).replace('.ifc', '.meta.json'))
fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(121, projection='3d'); static.plot_3d(vm, labels=labels, ax=ax1)
ax2 = fig.add_subplot(122); static.plot_graph(vm, labels=labels, ax=ax2)
plt.tight_layout(); plt.show()

In [ ]:
from viewer import view
# meta.json yolunu doğrudan labels olarak verebiliriz:
view(str(violated_path), labels=str(violated_path).replace('.ifc', '.meta.json'))

## Özet
- Baseline kurallara uygun (0 ihlal) → üzerine **kural tabanlı** 🔴/🟡/🟢 eklendi.
- `meta.json` ground-truth etiketleri taşır; viewer otomatik renklendirir.
- Tespit, enjekte edilen gerçek ihlallerle birebir eşleşir (decoy/compliant hariç).
- Sıradaki yöntemler: 3.2 tam LLM, 3.3 hibrit, 3.4 havuzdan seç→LLM→motor.